# Standardization

In [2]:
from pyspark.sql import SparkSession
from functools import reduce
from pyspark.sql.functions import col, year, to_date, lit, create_map, when
from pyspark.sql.functions import to_date, col, substring, concat_ws

import sys, os, time, getpass

sys.path.append("/home/tatiane/lib/")
import pessoal
from pessoal import *

spark = SparkSession.builder \
    .appName("SIM Processing") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [3]:
base_path = "/home/tatiane/repositorios/brazil-health-data-project/data_pipeline/data/processed/sim/conversion/one_file/"

In [4]:
dfs = []

for ano in os.listdir(base_path):
    caminho = os.path.join(base_path, ano)
    
    if os.path.isdir(caminho):
        df_ano = spark.read.parquet(caminho)
        
        # opcional: adicionar coluna do ano
        df_ano = df_ano.withColumn("ano", lit(ano))
        
        dfs.append(df_ano)

# unir tudo
df_sim = dfs[0]
for df in dfs[1:]:
    df_sim = df_sim.unionByName(df)

### Analisys

In [5]:
pessoal.completudeSchema(df_sim)

[Stage 5:======================================>                  (27 + 8) / 40]

Qtd. registros: 7600259 | Quantidade de colunas:  89
root
 |-- index: string (nullable = true)
 |-- ORIGEM: string (nullable = true)
 |-- TIPOBITO: string (nullable = true)
 |-- DTOBITO: string (nullable = true)
 |-- HORAOBITO: string (nullable = true)
 |-- NATURAL: string (nullable = true)
 |-- CODMUNNATU: string (nullable = true)
 |-- DTNASC: string (nullable = true)
 |-- IDADE: string (nullable = true)
 |-- SEXO: string (nullable = true)
 |-- RACACOR: string (nullable = true)
 |-- ESTCIV: string (nullable = true)
 |-- ESC: string (nullable = true)
 |-- ESC2010: string (nullable = true)
 |-- SERIESCFAL: string (nullable = true)
 |-- OCUP: string (nullable = true)
 |-- CODMUNRES: string (nullable = true)
 |-- LOCOCOR: string (nullable = true)
 |-- CODESTAB: string (nullable = true)
 |-- ESTABDESCR: integer (nullable = true)
 |-- CODMUNOCOR: string (nullable = true)
 |-- IDADEMAE: string (nullable = true)
 |-- ESCMAE: string (nullable = true)
 |-- ESCMAE2010: string (nullable = true)
 

In [6]:
df_sim.groupBy('ano').count().orderBy(F.col('ano').asc()).show()

[Stage 8:======================================================>  (38 + 2) / 40]

+--------+-------+
|     ano|  count|
+--------+-------+
|ano_2018|1316719|
|ano_2019|1349801|
|ano_2020|1556824|
|ano_2021|1832649|
|ano_2022|1544266|
+--------+-------+



### Structured

- Selecting and renaming variables

In [7]:
selected_cols = [
    "IDADE", "SEXO", "RACACOR", "ESTCIV",
    "CODMUNRES", "CODMUNOCOR",
    "DTOBITO", "DTNASC",
    "CAUSABAS", "CAUSABAS_O",
    "ASSISTMED", "NECROPSIA", "LOCOCOR"
]

df_sim = df_sim.select(*selected_cols)

In [8]:
mapping = {
    'IDADE':'idade',
    'SEXO':'sexo',
    'RACACOR':'raca_cor',
    'ESTCIV':'est_civ',
    'CODMUNRES':'cod_mun_res',
    'CODMUNOCOR':'cod_mun_ocor',
    'DTOBITO':'dt_obito',
    'DTNASC':'dt_nasc',
    'CAUSABAS':'causa_bas',
    'CAUSABAS_O':'causa_bas_o',
    'ASSISTMED':'assist_med',
    'NECROPSIA':'necropsia',
    'LOCOCOR':'local_ocor'
}

In [9]:
def rename_columns(df_sim, mapping):
    return reduce(
        lambda df_sim, col: df_sim.withColumnRenamed(col, mapping[col]),
        mapping,
        df_sim
    )

In [10]:
df_sim = rename_columns(df_sim, mapping)

In [11]:
pessoal.completudeSchema(df_sim)

Qtd. registros: 7600259 | Quantidade de colunas:  13
root
 |-- idade: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- raca_cor: string (nullable = true)
 |-- est_civ: string (nullable = true)
 |-- cod_mun_res: string (nullable = true)
 |-- cod_mun_ocor: string (nullable = true)
 |-- dt_obito: string (nullable = true)
 |-- dt_nasc: string (nullable = true)
 |-- causa_bas: string (nullable = true)
 |-- causa_bas_o: string (nullable = true)
 |-- assist_med: string (nullable = true)
 |-- necropsia: string (nullable = true)
 |-- local_ocor: string (nullable = true)



### Analisys

In [12]:
df_sim.show(truncate=False)

[Stage 15:===========================================>              (3 + 1) / 4]

+-----+----+--------+-------+-----------+------------+--------+--------+---------+-----------+----------+---------+----------+
|idade|sexo|raca_cor|est_civ|cod_mun_res|cod_mun_ocor|dt_obito|dt_nasc |causa_bas|causa_bas_o|assist_med|necropsia|local_ocor|
+-----+----+--------+-------+-----------+------------+--------+--------+---------+-----------+----------+---------+----------+
|457  |1   |2       |1      |290750     |290750      |21012019|18021961|I120     |I120       |1         |NULL     |1         |
|480  |2   |NULL    |1      |290750     |290750      |28012019|03071938|A419     |A419       |NULL      |NULL     |1         |
|473  |2   |4       |2      |290750     |290750      |27012019|15111945|E039     |E039       |NULL      |NULL     |1         |
|447  |1   |4       |2      |290750     |290750      |12012019|04111971|R220     |R220       |1         |2        |1         |
|472  |2   |4       |1      |290750     |290750      |10012019|12101946|C159     |C159       |NULL      |NULL  

In [13]:
list_col = ['sexo', 'raca_cor', 'est_civ', 'assist_med', 'necropsia', 'local_ocor']

for col in list_col:
    df_sim.groupBy(col).count().orderBy(F.col(col).asc()).show()

+----+-------+
|sexo|  count|
+----+-------+
|   0|   3142|
|   1|4213572|
|   2|3383545|
+----+-------+



+--------+-------+
|raca_cor|  count|
+--------+-------+
|    NULL| 183495|
|       1|3851412|
|       2| 626803|
|       3|  44648|
|       4|2869072|
|       5|  24829|
+--------+-------+



+-------+-------+
|est_civ|  count|
+-------+-------+
|   NULL| 391659|
|      1|1882344|
|      2|2457726|
|      3|1782215|
|      4| 517767|
|      5| 240638|
|      9| 327910|
+-------+-------+



+----------+-------+
|assist_med|  count|
+----------+-------+
|      NULL|2367933|
|         1|4081474|
|         2| 748192|
|         9| 402660|
+----------+-------+



+---------+-------+
|necropsia|  count|
+---------+-------+
|     NULL|2262852|
|        1| 858102|
|        2|4280636|
|        9| 198669|
+---------+-------+



+----------+-------+
|local_ocor|  count|
+----------+-------+
|         1|5090313|
|         2| 501452|
|         3|1510454|
|         4| 257520|
|         5| 235262|
|         6|    488|
|         9|   4770|
+----------+-------+



### Standardizing content

1. Date variables

In [14]:
df_sim_date = df_sim
colunas_data = ["dt_nasc", "dt_obito"]

for c in colunas_data:
    df_sim_date = df_sim_date.withColumn(
        f"{c}_format",
        to_date(F.col(c), "ddMMyyyy")
    )

In [15]:
df_sim_date.select('dt_nasc', 'dt_nasc_format', 'dt_obito', 'dt_obito_format').show()

+--------+--------------+--------+---------------+
| dt_nasc|dt_nasc_format|dt_obito|dt_obito_format|
+--------+--------------+--------+---------------+
|18021961|    1961-02-18|21012019|     2019-01-21|
|03071938|    1938-07-03|28012019|     2019-01-28|
|15111945|    1945-11-15|27012019|     2019-01-27|
|04111971|    1971-11-04|12012019|     2019-01-12|
|12101946|    1946-10-12|10012019|     2019-01-10|
|22061932|    1932-06-22|24012019|     2019-01-24|
|11121951|    1951-12-11|27012019|     2019-01-27|
|05031950|    1950-03-05|27012019|     2019-01-27|
|14071943|    1943-07-14|25012019|     2019-01-25|
|10121978|    1978-12-10|25012019|     2019-01-25|
|11121993|    1993-12-11|25012019|     2019-01-25|
|07111942|    1942-11-07|25012019|     2019-01-25|
|03041995|    1995-04-03|27012019|     2019-01-27|
|12111955|    1955-11-12|28012019|     2019-01-28|
|22081920|    1920-08-22|29012019|     2019-01-29|
|04051961|    1961-05-04|30012019|     2019-01-30|
|29031937|    1937-03-29|050220

2. Standardize encoded variables to text

In [16]:
map_raca_dict = {
    "1": "Branca",
    "2": "Preta",
    "3": "Amarela",
    "4": "Parda",
    "5": "Indígena"
}

map_sexo_dict = {
    "0": "Ignorado",
    "1": "Masculino",
    "2": "Feminino"
}

map_estciv_dict = {
    "1": "Solteiro",
    "2": "Casado",
    "3": "Viúvo",
    "4": "Separado/Divorciado",
    "5": "União estavel",
    "9": "Ignorado"
}

map_assistmed_dict = {
    "1": "Sim",
    "2": "Não",
    "9": "Ignorado"
}

map_necropsia_dict = {
    "1": "Sim",
    "2": "Não",
    "9": "Ignorado"
}

map_lococor_dict = {
    "1": "Hospital",
    "2": "Outro est. de saúde",
    "3": "Domicílio",
    "4": "Via pública",
    "5": "Outros",
    "6": "Aldeia indígena",
    "9": "Ignorado"
}

In [17]:
dic_mapeamentos = {
    "raca_cor": map_raca_dict,
    "sexo": map_sexo_dict,
    "est_civ": map_estciv_dict,
    "assist_med": map_assistmed_dict,
    "necropsia": map_necropsia_dict,
    "local_ocor": map_lococor_dict  
}

In [18]:
def aplicar_dicionarios(df, dic_mapeamentos):
    for coluna, dicionario in dic_mapeamentos.items():
        
        mapa = create_map([lit(x) for x in sum(dicionario.items(), ())])
        
        df = df.withColumn(
            f"{coluna}_desc",
            when(mapa[F.col(coluna)].isNull(), "NULL")
            .otherwise(mapa[F.col(coluna)])
        )
    
    return df

In [19]:
df_sim_encoded = aplicar_dicionarios(df_sim_date, dic_mapeamentos)

In [20]:
list_col2 = ['sexo', 'sexo_desc', 'raca_cor','raca_cor_desc', 'est_civ', 'est_civ_desc', 'assist_med', 'assist_med_desc',
             'necropsia', 'necropsia_desc', 'local_ocor', 'local_ocor_desc']

for col in list_col2:
    df_sim_encoded.groupBy(col).count().orderBy(F.col(col).asc()).show()

+----+-------+
|sexo|  count|
+----+-------+
|   0|   3142|
|   1|4213572|
|   2|3383545|
+----+-------+



+---------+-------+
|sexo_desc|  count|
+---------+-------+
| Feminino|3383545|
| Ignorado|   3142|
|Masculino|4213572|
+---------+-------+



+--------+-------+
|raca_cor|  count|
+--------+-------+
|    NULL| 183495|
|       1|3851412|
|       2| 626803|
|       3|  44648|
|       4|2869072|
|       5|  24829|
+--------+-------+



+-------------+-------+
|raca_cor_desc|  count|
+-------------+-------+
|      Amarela|  44648|
|       Branca|3851412|
|     Indígena|  24829|
|         NULL| 183495|
|        Parda|2869072|
|        Preta| 626803|
+-------------+-------+



+-------+-------+
|est_civ|  count|
+-------+-------+
|   NULL| 391659|
|      1|1882344|
|      2|2457726|
|      3|1782215|
|      4| 517767|
|      5| 240638|
|      9| 327910|
+-------+-------+



...

+---------+-------+
|necropsia|  count|
+---------+-------+
|     NULL|2262852|
|        1| 858102|
|        2|4280636|
|        9| 198669|
+---------+-------+



+--------------+-------+
|necropsia_desc|  count|
+--------------+-------+
|      Ignorado| 198669|
|          NULL|2262852|
|           Não|4280636|
|           Sim| 858102|
+--------------+-------+



+----------+-------+
|local_ocor|  count|
+----------+-------+
|         1|5090313|
|         2| 501452|
|         3|1510454|
|         4| 257520|
|         5| 235262|
|         6|    488|
|         9|   4770|
+----------+-------+



[Stage 69:=====================================================>  (38 + 2) / 40]

+-------------------+-------+
|    local_ocor_desc|  count|
+-------------------+-------+
|    Aldeia indígena|    488|
|          Domicílio|1510454|
|           Hospital|5090313|
|           Ignorado|   4770|
|Outro est. de saúde| 501452|
|             Outros| 235262|
|        Via pública| 257520|
+-------------------+-------+



3. Transformation of the age variable

In [21]:
# extracting unit and quantity
df_sim_age = df_sim_encoded.withColumn("unidade_idade", substring(F.col("idade"), 1, 1)) \
       .withColumn("quantidade_idade", substring(F.col("idade"), 2, 2))

# converting to numeric
df_sim_age = df_sim_age.withColumn("unidade_idade", F.col("unidade_idade").cast("int")) \
       .withColumn("quantidade_idade", F.col("quantidade_idade").cast("int"))

# converting everything to age in years
df_sim_age = df_sim_age.withColumn(
    "idade_anos",
    when(F.col("unidade_idade") == 1, F.col("quantidade_idade") / 525600)  # minutes
    .when(F.col("unidade_idade") == 2, F.col("quantidade_idade") / 8760)   # hours
    .when(F.col("unidade_idade") == 3, F.col("quantidade_idade") / 365.25)  # days
    .when(F.col("unidade_idade") == 4, F.col("quantidade_idade") / 12)      # months
    .when(F.col("unidade_idade") == 5, F.col("quantidade_idade"))           # yers
    .otherwise(None)
)

In [22]:
df_sim_age.select('idade_anos').show(10)

+------------------+
|        idade_anos|
+------------------+
|              4.75|
| 6.666666666666667|
| 6.083333333333333|
|3.9166666666666665|
|               6.0|
| 7.166666666666667|
| 5.583333333333333|
| 5.666666666666667|
|              6.25|
|3.3333333333333335|
+------------------+
only showing top 10 rows


In [23]:
# creation of age range
df_sim_age = df_sim_age.withColumn(
    "faixa_etaria",
    when(F.col("idade_anos") < 1, "0 ano")
    .when((F.col("idade_anos") >= 1) & (F.col("idade_anos") <= 4), "1-4")
    .when((F.col("idade_anos") <= 14), "5-14")
    .when((F.col("idade_anos") <= 24), "15-24")
    .when((F.col("idade_anos") <= 59), "25-59")
    .otherwise("60+")
)

# Creating ignored category
df_sim_age = df_sim_age.withColumn(
    "idade_anos",
    when(F.col("unidade_idade") == 9, None).otherwise(F.col("idade_anos"))
)

4. Removing columns and renaming them

In [24]:
df_sim_remove = df_sim_age.drop('sexo', 'raca_cor', 'est_civ', 'assist_med', 'necropsia', 'local_ocor', 'dt_obito', 'dt_nasc', 'idade', 'unidade_idade', 'quantidade_idade')

In [25]:
mapping = {
    'idade_anos':'idade',
    'sexo_desc':'sexo',
    'raca_cor_desc':'raca_cor',
    'est_civ_desc':'est_civ',
    'CODMUNRES':'cod_mun_res',
    'CODMUNOCOR':'cod_mun_ocor',
    'dt_obito_format':'dt_obito',
    'dt_nasc_format':'dt_nasc',
    'assist_med_desc':'assist_med',
    'necropsia_desc':'necropsia',
    'local_ocor_desc':'local_ocor'
}

In [26]:
df_sim_remove = rename_columns(df_sim_remove, mapping)

In [27]:
pessoal.completudeSchema(df_sim_remove)

Qtd. registros: 7600259 | Quantidade de colunas:  14
root
 |-- cod_mun_res: string (nullable = true)
 |-- cod_mun_ocor: string (nullable = true)
 |-- causa_bas: string (nullable = true)
 |-- causa_bas_o: string (nullable = true)
 |-- dt_nasc: date (nullable = true)
 |-- dt_obito: date (nullable = true)
 |-- raca_cor: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- est_civ: string (nullable = true)
 |-- assist_med: string (nullable = true)
 |-- necropsia: string (nullable = true)
 |-- local_ocor: string (nullable = true)
 |-- idade: double (nullable = true)
 |-- faixa_etaria: string (nullable = false)



5. Transforming core SIM variables

In [28]:
cid = spark.read \
    .option("sep", ";") \
    .option("header", True) \
    .option("encoding", "ISO-8859-1") \
    .csv("/home/tatiane/Downloads/CID-10-SUBCATEGORIAS.CSV")

pessoal.completude(cid)

Qtd. registros: 12451 | Quantidade de colunas:  9


In [29]:
cid.show(5)

+------+-------+---------+----------+--------------------+--------------------+-----+---------+----+
|SUBCAT|CLASSIF|RESTRSEXO|CAUSAOBITO|           DESCRICAO|          DESCRABREV|REFER|EXCLUIDOS| _c8|
+------+-------+---------+----------+--------------------+--------------------+-----+---------+----+
|  A000|   NULL|     NULL|      NULL|Cólera devida a V...|A00.0 Colera dev ...| NULL|     NULL|NULL|
|  A001|   NULL|     NULL|      NULL|Cólera devida a V...|A00.1 Colera dev ...| NULL|     NULL|NULL|
|  A009|   NULL|     NULL|      NULL|Cólera não especi...|     A00.9 Colera NE| NULL|     NULL|NULL|
|  A010|   NULL|     NULL|      NULL|       Febre tifóide| A01.0 Febre tifoide| NULL|     NULL|NULL|
|  A011|   NULL|     NULL|      NULL| Febre paratifóide A|A01.1 Febre parat...| NULL|     NULL|NULL|
+------+-------+---------+----------+--------------------+--------------------+-----+---------+----+
only showing top 5 rows


In [30]:
cid = cid.select("SUBCAT", "DESCRICAO")
cid = cid.withColumnRenamed("SUBCAT", "causa_bas")
cid = cid.withColumnRenamed("DESCRICAO", "nm_doenca")

In [32]:
df_sim_final = df_sim_remove.join(cid, "causa_bas", "inner")

In [33]:
df_sim_final = df_sim_final.withColumn(
    "houve_ajuste_causa",
    when(F.col("causa_bas") != F.col("causa_bas_o"), "Sim").otherwise("Não")
)

In [34]:
pessoal.completudeSchema(df_sim_final)

[Stage 83:=====================================================>  (38 + 2) / 40]

Qtd. registros: 7600016 | Quantidade de colunas:  16
root
 |-- causa_bas: string (nullable = true)
 |-- cod_mun_res: string (nullable = true)
 |-- cod_mun_ocor: string (nullable = true)
 |-- causa_bas_o: string (nullable = true)
 |-- dt_nasc: date (nullable = true)
 |-- dt_obito: date (nullable = true)
 |-- raca_cor: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- est_civ: string (nullable = true)
 |-- assist_med: string (nullable = true)
 |-- necropsia: string (nullable = true)
 |-- local_ocor: string (nullable = true)
 |-- idade: double (nullable = true)
 |-- faixa_etaria: string (nullable = false)
 |-- nm_doenca: string (nullable = true)
 |-- houve_ajuste_causa: string (nullable = false)



In [35]:
df_sim_final.limit(7).toPandas()

,causa_bas,cod_mun_res,cod_mun_ocor,causa_bas_o,dt_nasc,dt_obito,raca_cor,sexo,est_civ,assist_med,necropsia,local_ocor,idade,faixa_etaria,nm_doenca,houve_ajuste_causa
0,I120,290750,290750,I120,1961-02-18,2019-01-21,Preta,Masculino,Solteiro,Sim,NULL,Hospital,4.750000,5-14,Doença renal hipertensiva com insuficiência renal,Não
1,A419,290750,290750,A419,1938-07-03,2019-01-28,NULL,Feminino,Solteiro,NULL,NULL,Hospital,6.666667,5-14,Septicemia não especificada,Não
2,E039,290750,290750,E039,1945-11-15,2019-01-27,Parda,Feminino,Casado,NULL,NULL,Hospital,6.083333,5-14,Hipotireoidismo não especificado,Não
3,R220,290750,290750,R220,1971-11-04,2019-01-12,Parda,Masculino,Casado,Sim,Não,Hospital,3.916667,1-4,"Tumefação, massa ou tumoração localizadas da c...",Não
4,C159,290750,290750,C159,1946-10-12,2019-01-10,Parda,Feminino,Solteiro,NULL,NULL,Hospital,6.000000,5-14,"Neoplasia maligna do esôfago, não especificado",Não
5,I219,421900,421900,I219,1932-06-22,2019-01-24,Branca,Masculino,Casado,Sim,Sim,Domicílio,7.166667,5-14,Infarto agudo do miocárdio não especificado,Não
6,I260,420460,420460,I260,1951-12-11,2019-01-27,Preta,Masculino,Separado/Divorciado,Sim,Sim,Domicílio,5.583333,5-14,Embolia pulmonar com menção de cor pulmonale a...,Não


### Write

In [41]:
df_sim_final.coalesce(1) \
  .write \
  .mode("overwrite") \
  .parquet("/home/tatiane/repositorios/brazil-health-data-project/data_pipeline/data/processed/sim/standardization/")